# Limpeza dos dados

Carrega o CSV bruto exportado do Google Forms, renomeia colunas, padroniza bairros e salva um CSV limpo para os notebooks seguintes.

In [1]:
import pandas as pd

df = pd.read_csv("../dados/respostas_preliminares.csv")
df.head(2)
print(f"{df.shape[0]} respostas, {df.shape[1]} colunas")


161 respostas, 19 colunas


## 1. Renomear colunas

In [2]:
renomear = {
    df.columns[0]:  "timestamp",
    df.columns[1]:  "tipo_gerador",
    df.columns[2]:  "tipo_atividade",
    df.columns[3]:  "bairro_cidade",
    df.columns[4]:  "tipos_residuo",
    df.columns[5]:  "residuos_organicos",
    df.columns[6]:  "especificacao_residuos",
    df.columns[7]:  "origem_residuo",
    df.columns[8]:  "frequencia_descarte",
    df.columns[9]:  "quantidade_gerada",
    df.columns[10]: "destino_atual",
    df.columns[11]: "custo_destinacao",
    df.columns[12]: "potencial_reaproveitamento",
    df.columns[13]: "caracteristica_residuo",
    df.columns[14]: "tentativa_reutilizacao",
    df.columns[15]: "melhor_solucao",
    df.columns[16]: "num_pessoas",
    df.columns[17]: "conscientizacao",
    df.columns[18]: "interesse_aprender",
}

df = df.rename(columns=renomear)
df.columns.tolist()

['timestamp',
 'tipo_gerador',
 'tipo_atividade',
 'bairro_cidade',
 'tipos_residuo',
 'residuos_organicos',
 'especificacao_residuos',
 'origem_residuo',
 'frequencia_descarte',
 'quantidade_gerada',
 'destino_atual',
 'custo_destinacao',
 'potencial_reaproveitamento',
 'caracteristica_residuo',
 'tentativa_reutilizacao',
 'melhor_solucao',
 'num_pessoas',
 'conscientizacao',
 'interesse_aprender']

## 2. Padronização de bairros

Nesta etapa, o campo `bairro_cidade` é normalizado para reduzir inconsistências de preenchimento e melhorar a qualidade analítica da base.  
O procedimento inclui:

- Conversão de texto para minúsculas;
- Remoção de espaços excedentes e complementos textuais;
- Consolidação de variações ortográficas em categorias padronizadas para os bairros mais frequentes.

Registros que não correspondem aos padrões definidos são classificados na categoria `Outro`.

In [ ]:
def limpar_bairro(texto):
    """Normaliza o campo bairro/cidade para um rótulo padronizado."""
    if pd.isna(texto):
        return "Não informado"

    t = texto.strip().lower().replace("\n", " ")

    """
    Mapa: rótulo padronizado -> substrings que identificam o bairro.
    Ordem: bairros mais específicos antes dos genéricos.
    A busca é feita com `in`, então basta que a variante apareça em qualquer parte do texto original.
    """
    mapa = {
        "Jardim Itália":      ["jardim itália", "jardim italia"],
        "Jardim América":     ["jardim américa", "jardim america"],
        "Bela Vista":         ["bela vista"],
        "São Cristóvão":      ["são cristóvão", "sao cristovao", "santo antônio"],
        "Presidente Médici":  ["presidente médici", "presidente medici", "presidente medice"],
        "Vila Real":          ["vila real"],
        "Passo dos Fortes":   ["passo dos fortes"],
        "Coronel Freitas":    ["coronel freitas", "linha roncador"],
        "União do Oeste":     ["união do oeste", "uniao do oeste"],
        "Cristo Rei":         ["cristo rei"],
        "Frei Bruno":         ["frei bruno"],
        "Santa Maria":        ["santa maria"],
        "Santa Terezinha":    ["santa terezinha"],
        "Fernando Machado":   ["fernando machado"],
        "Maria Goretti":      ["maria goretti"],
        "Bom Senso":          ["bom senso"],
        "Germânico":          ["germanico", "germânico"],
        "Desbravador":        ["desbravador"],
        "Belvedere":          ["belvedere"],
        "Pinhalzinho":        ["pinhalzinho"],
        "Alvorada":           ["alvorada"],
        "Guarany":            ["guarany", "guarani"],
        "Primavera":          ["primavera"],
        "Saic":               ["saic"],
        "Flor":               ["flor"],
        "Líder":              ["líder", "lider"],
        "Efapi":              ["efapi"],
        "Xaxim":              ["xaxim"],
        "Centro":             ["centro"],
        "Chapecó (sem bairro)": ["chapecó", "chapeco"],
    }

    for padrao, variantes in mapa.items():
        for v in variantes:
            if v in t:
                return padrao

    return "Outro"


df["bairro_cidade"] = df["bairro_cidade"].apply(limpar_bairro)
df["bairro_cidade"].value_counts()

bairro_cidade
Xaxim                   20
Outro                   18
Efapi                   14
Centro                  14
Coronel Freitas          9
São Cristóvão            8
Chapecó (sem bairro)     8
União do Oeste           6
Primavera                6
Líder                    6
Bela Vista               5
Jardim Itália            5
Frei Bruno               4
Presidente Médici        4
Guarany                  4
Cristo Rei               3
Pinhalzinho              3
Passo dos Fortes         3
Vila Real                3
Desbravador              3
Flor                     2
Jardim América           2
Belvedere                2
Alvorada                 2
Fernando Machado         1
Bom Senso                1
Maria Goretti            1
Germânico                1
Saic                     1
Santa Terezinha          1
Santa Maria              1
Name: count, dtype: int64

## 3. Padronização de `destino_atual`

O campo `destino_atual` reúne respostas fechadas e respostas livres. Nesta etapa, essas respostas são consolidadas em categorias principais para reduzir a fragmentação visual e tornar os resultados mais interpretáveis.

A ordem da busca importa. Em respostas que combinam mais de uma categoria, prevalece a primeira categoria encontrada.

In [4]:
import unicodedata

print("Antes da limpeza de destino_atual:")
print(df["destino_atual"].value_counts(dropna=False).to_string())

def normalizar_texto(texto):
    texto = str(texto).strip().lower().replace("\n", " ")
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")

def limpar_destino(texto):
    """Normaliza o campo destino_atual para categorias principais."""
    if pd.isna(texto):
        return "Não informado"

    t = normalizar_texto(texto)

    mapa = {
        "Compostagem": ["compost", "minhoca"],
        "Reaproveitamento": ["galinha", "porco", "lavagem", "alimentacao", "reaprov", "enterro", "doacao", "dou pras"],
        "Coleta seletiva": ["seletiva"],
        "Coleta pública comum": ["publica"],
        "Descarte": ["descarte", "aterro", "queima"],
        "Não sei": ["nao sei"],
    }

    for padrao, variantes in mapa.items():
        for v in variantes:
            if v in t:
                return padrao

    return texto.strip()

df["destino_atual"] = df["destino_atual"].apply(limpar_destino)

print("\nDepois da limpeza de destino_atual:")
print(df["destino_atual"].value_counts(dropna=False).to_string())


Antes da limpeza de destino_atual:
destino_atual
Coleta pública comum                                                                                           84
Coleta seletiva                                                                                                32
Compostagem                                                                                                    21
Reaproveitamento interno                                                                                       11
Aterro industrial                                                                                               2
Descarte                                                                                                        2
Coleta seletiva e compostagem                                                                                   1
Dou pras galinhas ou enterro                                                                                    1
Compostagem / coleta pública           

**Nota sobre abrangência geográfica:** das 161 respostas, aproximadamente 18 foram classificadas como "Outro". A maioria corresponde a municípios vizinhos (Ipuaçu, Planalto Alegre, Cunha Porã, Santiago do Sul, Seara, Xanxerê) ou localidades fora da região (Guarapuava/PR, Vilhena/RO, Rio de Janeiro, Sorriso/MT, Maravilha/SC). Esses registros são mantidos na base para preservar a integridade dos dados, mas evidenciam que o alcance do formulário ultrapassou a região de Chapecó. A análise exploratória nos notebooks seguintes foca na amostra completa.

## 4. Salvar CSV limpo

In [5]:
caminho_saida = "../dados/respostas_limpo.csv"
df.to_csv(caminho_saida, index=False)
print(f"Salvo em {caminho_saida} -- {df.shape[0]} linhas, {df.shape[1]} colunas")

Salvo em ../dados/respostas_limpo.csv -- 161 linhas, 19 colunas